In [ ]:
import torch
import torch.nn as nn 
import torch.nn.functional as F
import numpy as np

n = 6 # source_length
m = 8 # target_length
d_model = 512
d_k = 64
d_v = 64
numRepeatedUnits = 6
numBlocks = 6
vocabSize = 512
maxLength = 256

#
# 
class simpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.ff1 = nn.Linear(d_model, 4*d_model)
        self.ff2 = nn.Linear(4*d_model, d_model)

    def forward(self, x):
        x = self.ff1(x)
        x = torch.relu(x)
        x = self.ff2(x)
        return x

class crossAttention(nn.Module):
    def __init__(self, masked=False):
        super().__init__()
        self.Wk = torch.nn.Linear(d_model, d_k, bias = False) # torch.nn.Parameter(torch.randn(d_model, d_k) * 0.1)
        self.Wq = torch.nn.Linear(d_model, d_k, bias = False) # torch.nn.Parameter(torch.randn(d_model, d_k) * 0.1)
        self.Wv = torch.nn.Linear(d_model, d_k, bias = False) # torch.nn.Parameter(torch.randn(d_model, d_v) * 0.1)
        self.masked = masked

    def forward(self, x, y):

        q = self.Wq(x) # torch.matmul(x, self.Wq)  # x has size of source_length x d_model
        k = self.Wk(y) # torch.matmul(y, self.Wk)  # y has size of target_length x d_model
        v = self.Wv(y)  # torch.matmul(y, self.Wv)
        inside = torch.matmul(q,k.T)  # inside of attention weights has the dimension of (query x keys)
        # print(inside.shape)
        

        if self.masked:
                # If masked, then we need to make sure that
                # the query token cannot attend to future tokens. 
                # This means that the we set "inside" to be -inf for all future tokens.
                # set anything that is above the diagonal to be -inf
                mask = torch.triu(torch.ones(inside.shape), diagonal=1).bool()
                inside = inside.masked_fill(mask, float('-inf'))
            

        # Normalize along the last dimension, the key
        # Reason: Each query needs to see which key is the most important, meaning each query needs to be independent. 
        # The probabilities add up to one for every query.
        # Means, for each query, we take all the keys and take the average. This means that dimension 1 is the keys. 

        attn_weights = torch.softmax(inside / np.sqrt(d_k), -1) 
        attn = torch.matmul(attn_weights, v)

        return attn
        # attention: shape of m x m 

class encoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.selfAttention = crossAttention()
        self.layerNorm1 = nn.LayerNorm(d_model) # normalize over d_model dimension, think about it.
        self.Wo = torch.nn.Linear(d_k, d_model, bias = False) # instead of torch.nn.Parameter(torch.empty(d_k, d_model))
        # torch.nn.init.xavier_uniform_(self.Wo)

        self.simpleNN = simpleNN()
        self.layerNorm2 = nn.LayerNorm(d_model) # normalize over d_model dimension, think about it.
        

    def forward(self, x):
        delta_x_1 = self.selfAttention(x, x)
        
        x = x+self.Wo(delta_x_1)
        x = self.layerNorm1(x)

        delta_x_2 = self.simpleNN(x)
        x = x + delta_x_2
        x = self.layerNorm2(x)
        return x 

class decoderBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.maskedSelfAttention = crossAttention(masked=True)
        self.layerNorm1 = nn.LayerNorm(d_model) # normalize over d_model dimension, think about it.
        self.Wo1 = torch.nn.Linear(d_k, d_model, bias = False) #torch.nn.Parameter(torch.randn(d_k, d_model) * 0.1)

        self.crossAttention = crossAttention()
        self.layerNorm2 = nn.LayerNorm(d_model) # normalize over d_model dimension, think about it.
        self.Wo2 = torch.nn.Linear(d_k, d_model, bias = False) # torch.nn.Parameter(torch.randn(d_k, d_model) * 0.1)

        self.simpleNN = simpleNN()
        self.layerNorm3 = nn.LayerNorm(d_model) # normalize over d_model dimension, think about it.

    def forward(self, x, y):
        delta_y_1 = self.maskedSelfAttention(y, y)
        # y = y+torch.matmul(delta_y_1, self.Wo1)
        y = y+self.Wo1(delta_y_1)
        y = self.layerNorm1(y)

        delta_y_2 = self.crossAttention(y, x) # the "query" is the first parameter. the query is the output embedding
        # Here, we're using the output sequence as the query, the input is the key
        # overall, the query in output is finding tokens in the input to attend over.
        y = y+self.Wo2(delta_y_2) # y=y+torch.matmul(delta_y_2, self.Wo2)
        y = self.layerNorm2(y)

        y= self.simpleNN(y)

        return y
        

class positionalEncoder(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        # Dummy positional encoding that add 
        for i in range(0,d_model//2):
            for pos in range(0,len(x)):
                with torch.no_grad():
                    x[pos][2*i] += np.sin(pos / 10e4 ** (2*(i / d_model)))
                    x[pos][2*i+1] += np.cos(pos / 10e4 ** (2*(i / d_model)))
        return x

class Attention(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoderBlocks = nn.ModuleList([encoderBlock() for _ in range(numBlocks)])
        self.decoderBlocks = nn.ModuleList([decoderBlock() for _ in range(numBlocks)])
        self.positionalEncoder = positionalEncoder()
        self.linear = nn.Linear(d_model, vocabSize)

    def forward(self, x, y):
        assert x.shape == (n, d_model)
        # assert y.shape == (m, d_model)
        x = self.positionalEncoder(x)
        for block in self.encoderBlocks:
            x = block(x)
        assert x.shape == (n, d_model)

        y = self.positionalEncoder(y)
        # print(y.shape)
        for block in self.decoderBlocks:
            y = block(x, y)
        # print(y.shape)
        assert y.shape == (m, d_model)

        y = torch.softmax(self.linear(y), -1)
        assert y.shape == (m, vocabSize)

        return y
        

# Instantiate the network
model = Attention()
# x = torch.nn.Parameter(torch.randn(n, d_model) * 0.1)
# y = torch.nn.Parameter(torch.randn(m, d_model) * 0.1)


In [330]:
!pip install scikit-learn


[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [278]:
import pandas as pd
from sklearn.model_selection import train_test_split

df_all = pd.read_parquet('/Users/michael/Documents/Projects/attention/data/0000-2.parquet')
df_all.columns = ['new','old']


df_all


,new,old
0,"When someone says 'She's thirsty, ain't she?',...","When one remarks, ""She doth crave attention, d..."
1,Saying 'He's flexing those dollar bills.' mean...,"To utter, ""He doth flaunt his gold coins, veri..."
2,The phrase 'That's some tea.' refers to juicy ...,"The saying, 'That's a tale most intriguing.' s..."
3,Exclaiming 'It's lit!' means the situation is ...,Declaring 'It's a merry revel!' signifies a jo...
4,"When someone uses 'Throwing shade', they're su...","When one says ""Casting aspersions"", they art d..."
...,...,...
269,I love that for you. (Said with sarcasm),I doth adore that for thee. (Said with irony)
270,Mood. It's been a long week.,"Aye, I can relate. This week hath been arduous."
271,That celebrity is a snack!,That celebrity is a delectable morsel!
272,I stan that singer; their music is amazing.,I am an ardent admirer of that minstrel; their...


In [321]:
def truncate_pad_word_token(lst):
    if len(lst) > maxLength:
        lst = lst[0:maxLength]
        
    else:
        lst = lst + ['<PAD>'] * (maxLength-len(lst))
    
    return lst

# def convert_to_id(lst):
#     return [new_dict[i] for i in lst]
    


In [322]:
import re
from collections import Counter
import random
random.seed(5)


# Splits data into 80% train and 20% test with a random shuffle
df_train, df_test = train_test_split(df_all, test_size=0.2, random_state=42)

# If you need to add the 'train_test_split' column back to your original DataFrame
df_all['train_test_split'] = 'train'
df_all.loc[df_test.index, 'train_test_split'] = 'test'


df_train = df_all.query("train_test_split == 'train'")
df_test = df_all.query("train_test_split == 'test'")
# sample something that is of low frequency and replace with <unk>
MIN_FREQ = 2  # tune this based on your corpus size

def replace_rare(tokens, freq_dict, min_freq=MIN_FREQ):
    return ['<UNK>' if freq_dict[tok] <= min_freq and random.random() > 0.8 else tok for tok in tokens]

# Build a tokenizer
freq_dicts = {}
vocab_sets = {}
for df in [df_train,df_test]:
    for col in ['new', 'old']:
        df[f'{col}'] = df[col].astype(str)
        df[f'{col}'] = df[f'{col}'].str.replace(r'\s+', ' ', regex=True).str.replace('\xa0', ' ', regex=False)

        df[f'{col}'] = df[f'{col}'].apply(
            lambda x: re.sub(r'[^a-zA-Z\s]', '', str(x).lower().strip())
        )

        df[col] = '<START> ' + df[col] + ' <END>'
        df[f'{col}_list'] = df[f'{col}'].apply(
            lambda x: [w for w in x.split(' ') if w != '']
        )


        if df is df_train:
            tr_list = df[f'{col}_list'].sum()
            freq_dicts[col] = Counter(tr_list)

            df[f'{col}_list'] = df[f'{col}_list'].apply(lambda toks: replace_rare(toks, freq_dicts[col]))
            vocab_sets[col] = set(df[f'{col}_list'].sum())

new_dict, old_dict = freq_dicts['new'], freq_dicts['old']
new_vocab, old_vocab = vocab_sets['new'], vocab_sets['old']

# dictionary to ID-izer
constant_dict_id = {
    0: '<START>',
    1: '<END>',
    2: '<PAD>',
    3: '<UNK>'
}
new_dict_id = {i+4: word for i, word in enumerate(new_dict.keys())} | constant_dict_id
old_dict_id = {i+4: word for i, word in enumerate(old_dict.keys())} | constant_dict_id
new_dict_id_rev = defaultdict(lambda:3,{v: k for k, v in new_dict_id.items()}) # returns 3 ('<UNK>' by default)
old_dict_id_rev = defaultdict(lambda:3,{v: k for k, v in old_dict_id.items()}) # returns 3 ('<UNK>' by default)

from collections import defaultdict
dict_id = {'new': new_dict_id, 'old': old_dict_id}
dict_id_rev = {'new': new_dict_id_rev, 'old': old_dict_id_rev}
# d
for i in ['old','new']:
    df_train[f'{i}_list'] = df_train[f'{i}_list'].apply(truncate_pad_word_token)
    df_test[f'{i}_list'] = df_test[f'{i}_list'].apply(truncate_pad_word_token)

    df_train[f'tokenized_{i}_list'] = df_train[f'{i}_list'].apply(lambda lst: [dict_id_rev[i][j] for j in lst])
    df_test[f'tokenized_{i}_list'] = df_test[f'{i}_list'].apply(lambda lst: [dict_id_rev[i][j] for j in lst])


In [323]:
df_train

,new,old,train_test_split,new_list,old_list,tokenized_old_list,tokenized_new_list
0,<START> when someone says shes thirsty aint sh...,<START> when one remarks she doth crave attent...,train,"[<START>, when, someone, says, shes, thirsty, ...","[<START>, when, one, remarks, she, doth, crave...","[0, 5, 6, 7, 8, 9, 10, 11, 9, 8, 12, 13, 14, 1...","[0, 5, 6, 7, 8, 9, 10, 11, 12, 13, 8, 3, 15, 1..."
1,<START> saying hes flexing those dollar bills ...,<START> to utter he doth flaunt his gold coins...,train,"[<START>, <UNK>, hes, flexing, those, dollar, ...","[<START>, to, utter, he, doth, <UNK>, his, gol...","[0, 20, 21, 22, 9, 3, 24, 25, 26, 27, 28, 20, ...","[0, 3, 18, 19, 20, 21, 3, 23, 18, 24, 25, 26, ..."
2,<START> the phrase thats some tea refers to ju...,<START> the saying thats a tale most intriguin...,train,"[<START>, the, phrase, thats, some, tea, refer...","[<START>, the, saying, thats, a, tale, most, i...","[0, 32, 33, 34, 35, 36, 37, 38, 39, 40, 35, 41...","[0, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38..."
3,<START> exclaiming its lit means the situation...,<START> declaring its a merry revel signifies ...,train,"[<START>, exclaiming, its, lit, means, the, si...","[<START>, declaring, its, a, merry, revel, <UN...","[0, 45, 46, 35, 47, 48, 3, 35, 3, 3, 1, 2, 2, ...","[0, 40, 41, 42, 23, 28, 43, 44, 3, 37, 46, 1, ..."
4,<START> when someone uses throwing shade theyr...,<START> when one says casting aspersions they ...,train,"[<START>, when, someone, uses, throwing, shade...","[<START>, when, one, says, casting, aspersions...","[0, 5, 6, 52, 53, 54, 13, 55, 56, 57, 58, 59, ...","[0, 5, 6, 47, 48, 49, 12, 50, 51, 3, 37, 53, 1..."
...,...,...,...,...,...,...,...
268,<START> karen demanded to speak with the manag...,<START> karen didst insist on an audience with...,train,"[<START>, <UNK>, <UNK>, to, speak, with, the, ...","[<START>, karen, didst, insist, on, an, audien...","[0, 917, 499, 918, 130, 94, 919, 207, 32, 920,...","[0, 3, 3, 34, 506, 179, 28, 663, 226, 1, 2, 2,..."
270,<START> mood its been a long week <END>,<START> aye i can relate this week hath been a...,train,"[<START>, mood, its, been, a, <UNK>, week, <EN...","[<START>, aye, i, can, relate, this, week, hat...","[0, 921, 106, 857, 922, 89, 474, 70, 843, 923,...","[0, 257, 41, 284, 111, 3, 347, 1, 2, 2, 2, 2, ..."
271,<START> that celebrity is a snack <END>,<START> that celebrity is a delectable morsel ...,train,"[<START>, that, celebrity, is, a, snack, <END>...","[<START>, that, celebrity, is, a, delectable, ...","[0, 42, 924, 28, 35, 925, 926, 1, 2, 2, 2, 2, ...","[0, 59, 665, 44, 111, 563, 1, 2, 2, 2, 2, 2, 2..."
272,<START> i stan that singer their music is amaz...,<START> i am an ardent admirer of that minstre...,train,"[<START>, i, stan, that, singer, their, music,...","[<START>, i, am, an, <UNK>, admirer, of, that,...","[0, 106, 168, 94, 3, 928, 40, 42, 929, 426, 93...","[0, 93, 185, 59, 666, 309, 667, 44, 311, 1, 2,..."


In [317]:
import torch
import torch.nn as nn 

new_vocab_size = len(new_dict_id)
old_vocab_size = len(old_dict_id)
new_embedding = nn.Embedding(new_vocab_size, d_model)
old_embedding = nn.Embedding(old_vocab_size, d_model)
# e.g. 
# new_embedding(torch.tensor(5))  # grabs row 5

In [326]:
from torch.utils.data import DataLoader, TensorDataset

train_dataset = TensorDataset(
    torch.tensor(df_train['tokenized_old_list'].tolist(), dtype = torch.long),
    torch.tensor(df_train['tokenized_new_list'].tolist(), dtype = torch.long),
        
)

train_loader = DataLoader(train_dataset, batch_size = 32, shuffle =True)
# train_old_ids_tensor = torch.tensor(df_train['id_old_list'].tolist(), dtype=torch.long)
# train_new_ids_tensor = torch.tensor(df_train['id_new_list'].tolist(), dtype=torch.long)

# old_embedding(train_old_ids_tensor)


In [327]:
train_loader

In [90]:
for df in [df_test]:
    for col in ['new', 'old']:
        df[f'{col}_response'] = df[col].astype(str)
        df[f'{col}_dialog'] = df[f'{col}_response'].str.replace(r'\s+', ' ', regex=True).str.replace('\xa0', ' ', regex=False)

        df[f'{col}_list'] = df[f'{col}_dialog'].apply(
            lambda x: [w for w in re.sub(r'[^a-zA-Z\s]', '', str(x).lower().strip()).split(' ') if w != '']
        )

In [91]:
df_train['old_list']

0      [<START>, when, one, remarks, she, doth, crave...
1      [<START>, to, utter, he, doth, flaunt, his, go...
2      [<START>, the, saying, thats, a, tale, most, i...
3      [<START>, declaring, its, a, merry, revel, <UN...
4      [<START>, when, one, says, casting, aspersions...
                             ...                        
268    [<START>, karen, didst, insist, on, an, audien...
270    [<START>, aye, i, can, relate, this, week, hat...
271    [<START>, that, celebrity, is, a, delectable, ...
272    [<START>, i, am, an, <UNK>, admirer, of, that,...
273    [<START>, that, excuse, doth, sound, suspiciou...
Name: old_list, Length: 219, dtype: object

In [ ]:
# Keep token IDs in DataFrame
df_train['id_old_list']
df_train['id_new_list']

# During training, embed on the fly:
class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, df, old_embedding, new_embedding, old_dict_rev, new_dict_rev):
        self.df = df
        self.old_embedding = old_embedding
        self.new_embedding = new_embedding
    
    def __getitem__(self, idx):
        old_ids = torch.tensor(self.df.iloc[idx]['id_old_list'], dtype=torch.long)
        new_ids = torch.tensor(self.df.iloc[idx]['id_new_list'], dtype=torch.long)
        
        old_emb = self.old_embedding(old_ids)
        new_emb = self.new_embedding(new_ids)
        
        return old_emb, new_emb

In [331]:
NUM_EPOCH = 50
# # training
# X = torch.randn(n, d_model)
# Y = torch.randn(m, d_model)

model = Attention()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(NUM_EPOCH):
    total_loss = 0

    for batch_old_ids, batch_new_ids in train_loader:
        # Embed token
        X = old_embedding(batch_old_ids)
        Y = old_embedding(batch_new_ids)

        optimizer.zero_grad()

        logits = model(X,Y) # produces probability over each token

        # don't understand
        loss = F.cross_entropy(
            logits.reshape(-1, vocabSize),
            batch_new_ids.reshape(-1)
        )
        
        # loss = F.cross_entropy(logits, Y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    if epoch % 10 == 0:
        print(epoch, loss.item())


/var/folders/c6/jslqh2g54lq5hvqmtgmnqx2w0000gn/T/ipykernel_1750/3035434138.py:43: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  inside = torch.matmul(q,k.T)


RuntimeError: The size of tensor a (32) must match the size of tensor b (64) at non-singleton dimension 0

In [ ]:
# Question: Why is dk, dv equal?
# Why is dk and dv the same dimension? well, because we want to dot product them. 